In [3]:
import pandas as pd

datos = pd.read_csv("../data/processed/datos_procesados_2021.csv")

print("Datos:", datos.shape)

Datos: (107787, 42)



Pregunta 1 (3 pts): Retención de clientes
¿Cuántos VINs únicos visitaron el taller tanto en Q1 como en Q2? Lista los primeros 10 VINs y su número total de visitas en el semestre.

In [5]:
# Convertir columnas para aplicar correctamente los filtros
ventas = pd.to_numeric(
    datos["Venta$"].astype("string")
    .str.replace(r"[$,]", "", regex=True)
    .str.strip(),
    errors="coerce"
)
ventas_netas = pd.to_numeric(datos["VentasNetas"], errors="coerce")
vin = datos["Vin"].astype("string").str.strip()

# Aplicar las tres condiciones solicitadas
filtro = (
    datos["TipoRegistro"].eq("Registro de venta")
    & ventas_netas.eq(1)
    & ventas.gt(0)
    & vin.notna()
    & vin.ne("")
    & datos["Fuente"].isin(["Q1", "Q2"])
)

datos_filtrados = datos.loc[filtro, ["Fuente", "NumeroOT"]].copy()
datos_filtrados["Vin"] = vin.loc[filtro]

def mostrar_top10(df, nombre):
    resumen = (
        df.groupby("Vin")
        .agg(Visitas=("NumeroOT", "nunique"))
        .sort_values(["Visitas", "Vin"], ascending=[False, True])
    )

    print(f"\n{nombre} — VINs únicos: {len(resumen)}")
    display(resumen.head(10).reset_index())

# Top 10 conjunto de ambos trimestres
mostrar_top10(datos_filtrados, "Q1 + Q2")

# Top 10 de cada trimestre
mostrar_top10(datos_filtrados[datos_filtrados["Fuente"] == "Q1"], "Q1")
mostrar_top10(datos_filtrados[datos_filtrados["Fuente"] == "Q2"], "Q2")


Q1 + Q2 — VINs únicos: 68100


,Vin,Visitas
0,3KPA34AC8JE108967,11
1,2A4RR6DG9BR760969,9
2,3N1CN7AD2LK414311,8
3,5KBRL6889KB800670,8
4,1GKKR9KD4EJ125480,7
5,3G1J86CC1GS605484,7
6,3KPFK4A78JE149997,7
7,3N1CN7AD9LK411485,7
8,3N1CN8AE3LL906340,7
9,3TMAZ5CN6LM124144,7



Q1 — VINs únicos: 38030


,Vin,Visitas
0,3KPFK4A78JE149997,7
1,KMHC75LC0KU156042,6
2,KNDJP3A48K7671204,6
3,LSGHD52H2LD083455,6
4,1GTU99EL6KZ316547,5
5,2HGFC3277KH850558,5
6,3C6SRBDG5KG561656,5
7,3HGRM3850FG005153,5
8,3KPA34AC6JE020273,5
9,3KPC25B51KE042605,5



Q2 — VINs únicos: 37677


,Vin,Visitas
0,3KPA34AC8JE108967,10
1,3GNCJ7CE6JL261123,6
2,KL8CD6AD6FC777577,6
3,19UUB1654HA900038,5
4,1G1F91R78J0119978,5
5,2A4RR6DG9BR760969,5
6,3G1J85CC6GS591544,5
7,3G1J86CC1GS605484,5
8,3HGRU5893KM009711,5
9,3KPF14BD7LE185724,5


Pregunta 2 (5 pts): Distribución de órdenes
Calcula el porcentaje de órdenes de trabajo por Sucursal y por Tipo de Orden. Presenta el resultado en una tabla pivote que muestre: Sucursal | Tipo Orden | % del Total de Órdenes.

In [7]:
# Convertir las columnas para poder aplicar los filtros
ventas = pd.to_numeric(
    datos["Venta$"].astype("string")
    .str.replace(r"[$,]", "", regex=True)
    .str.strip(),
    errors="coerce"
)
ventas_netas = pd.to_numeric(datos["VentasNetas"], errors="coerce")

# Deben cumplirse las tres condiciones
filtro = (
    datos["TipoRegistro"].eq("Registro de venta")
    & ventas_netas.eq(1)
    & ventas.gt(0)
)

datos_tabla = datos.loc[filtro].copy()
datos_tabla["Sucursal"] = datos_tabla["Nombre"].fillna("Sin Definir")
datos_tabla["TipoOrden"] = datos_tabla["TipoOrden"].fillna("Sin Definir")

# Evitar contar varias filas de una misma orden
ordenes = datos_tabla.drop_duplicates(
    subset=["Client", "Branch", "NumeroOT"]
)

total_ordenes = len(ordenes)

if total_ordenes == 0:
    raise ValueError("No hay órdenes que cumplan las tres condiciones.")

tabla_pivote = ordenes.pivot_table(
    index=["Sucursal", "TipoOrden"],
    values="NumeroOT",
    aggfunc="count",
    fill_value=0
).rename(columns={"NumeroOT": "Órdenes"})

tabla_pivote["% del Total de Órdenes"] = (
    tabla_pivote["Órdenes"] / total_ordenes * 100
).round(2)

tabla_pivote = (
    tabla_pivote[["% del Total de Órdenes"]]
    .reset_index()
    .rename(columns={"TipoOrden": "Tipo Orden"})
    .sort_values("% del Total de Órdenes", ascending=False)
)

display(tabla_pivote)

,Sucursal,Tipo Orden,% del Total de Órdenes
59,Sucursal K,PUBLICO,6.53
42,Sucursal G,PUBLICO,5.30
38,Sucursal F,PUBLICO,5.07
84,Sucursal P,PUBLICO,4.65
88,Sucursal Q,PUBLICO,4.39
...,...,...,...
75,Sucursal N,Sin Definir,0.00
94,Sucursal R,Sin Definir,0.00
70,Sucursal M,Sin Definir,0.00
102,Sucursal U,ASEGURADORA,0.00
